# Alexa Review Sentiment Classifier

A sentiment classifier on the Amazon Alexa reviews dataset. The target `feedback`
(1 positive, 0 negative) is predicted from the free text in `verified_reviews`.

This notebook is the narrative report. It does no modeling of its own: it calls into
`src/` for curation, preprocessing, evaluation, thresholding, interpretation, and fairness,
and model selection runs through the config-driven harness (logged to MLflow). The theme is
label skepticism: `feedback` is a hard threshold on the star `rating`, so "sentiment" is a
thresholded star count, and the hard cases are text/rating disagreements.

## Imports

Everything imported once. The repo root is on the path so `src` resolves.

In [1]:
import os
import sys

REPO_ROOT = os.path.abspath("..")
sys.path.insert(0, REPO_ROOT)

import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.dummy import DummyClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_curve
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import Pipeline

from src.data.load import curate, load_raw
from src.evaluation.cv import cross_validate_negative
from src.evaluation.error_analysis import disagreements
from src.evaluation.fairness import separation_by_group
from src.evaluation.interpret import top_coefficients
from src.evaluation.metrics import evaluate, negative_scores
from src.evaluation.threshold import choose_threshold, evaluate_at_threshold, sweep
from src.features.text import build_vectorizer

warnings.simplefilter("ignore", ConvergenceWarning)
RANDOM_STATE = 42
RAW = os.path.join(REPO_ROOT, "data", "raw", "amazon_alexa.tsv")
pd.set_option("display.max_colwidth", 90)

## Data and label

We load and curate through `src.data` (drops empty reviews and exact duplicates, validated
by a Pandera schema), then confirm what the label is.

In [2]:
df, report = curate(load_raw(RAW))
print(report)
crosstab = pd.crosstab(df["rating"], df["feedback"], margins=True)
rule_holds = ((df["rating"].isin([1, 2])) == (df["feedback"] == 0)).all()
print("feedback == 1 iff rating >= 3 holds for all rows:", bool(rule_holds))
crosstab

CurationReport(raw_rows=3150, dropped_blank=80, dropped_duplicates=686, curated_rows=2384, negatives=205)
feedback == 1 iff rating >= 3 holds for all rows: True


feedback,0,1,All
rating,,,
1,129,0,129
2,76,0,76
3,0,106,106
4,0,340,340
5,0,1733,1733
All,205,2179,2384


The classes are heavily imbalanced toward positive (about 8.6% negative), which is why
accuracy is never the headline and why the split below is stratified: stratifying keeps the
rare negative prevalence in both the train and the held-out test set.

In [3]:
fig = px.histogram(df, x="feedback", title="Label balance: heavily positive",
                   labels={"feedback": "feedback (0 = negative, 1 = positive)"})
fig.update_layout(bargap=0.2)
fig

In [4]:
train_df, test_df = train_test_split(
    df, test_size=0.20, stratify=df["feedback"], random_state=RANDOM_STATE
)
X_train, y_train = train_df["verified_reviews"], train_df["feedback"]
X_test, y_test = test_df["verified_reviews"], test_df["feedback"]
print("train:", len(train_df), " test:", len(test_df),
      " test negative share: %.3f" % (1 - y_test.mean()))

train: 1907  test: 477  test negative share: 0.086


## Baseline floor

A do-nothing majority predictor, scored through the shared `evaluate`, negative class
first. Every model below is read against this.

In [5]:
dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
dummy_metrics = evaluate(y_test, dummy.predict(X_test), neg_score=negative_scores(dummy, X_test))
{k: round(v, 3) for k, v in dummy_metrics.items() if k != "confusion_matrix"}

{'neg_precision': 0.0,
 'neg_recall': 0.0,
 'neg_f1': 0.0,
 'specificity': 1.0,
 'pr_auc': 0.086}

## The selected control

Selection is done by the harness on PR-AUC (threshold-independent negative-class ranking),
which avoids the degenerate high-recall model that selecting on raw recall produced. The
winner is logistic regression on TF-IDF with an L2 penalty at `C = 10`, unweighted. Here we
present that model through the shared cross-validation and evaluation. (The single fit uses
`liblinear` for speed; the harness compares L1 and L2 on `saga` across the grid.)

In [6]:
control = Pipeline([
    ("tfidf", build_vectorizer("tfidf", min_df=2)),
    ("clf", LogisticRegression(penalty="l2", C=10, solver="liblinear",
                               max_iter=2000, random_state=RANDOM_STATE)),
])
cv = cross_validate_negative(control, X_train, y_train, n_splits=5, n_repeats=3, seed=RANDOM_STATE)
control.fit(X_train, y_train)
neg_test = negative_scores(control, X_test)
holdout = evaluate(y_test, control.predict(X_test), neg_score=neg_test)

pd.DataFrame([
    {"model": "Dummy (most_frequent)", **{k: v for k, v in dummy_metrics.items() if k != "confusion_matrix"}},
    {"model": "TF-IDF + LogReg L2 (0.5 threshold)", **{k: v for k, v in holdout.items() if k != "confusion_matrix"}},
]).set_index("model").round(3)

,neg_precision,neg_recall,neg_f1,specificity,pr_auc
model,,,,,
Dummy (most_frequent),0.000,0.000,0.000,1.000,0.086
TF-IDF + LogReg L2 (0.5 threshold),0.714,0.244,0.364,0.991,0.575


## Operating point

The default 0.5 threshold is far too conservative under this imbalance (it almost never
predicts negative). Selection and operating point are separate decisions: we pick the
threshold on out-of-fold training predictions (never the test set) to reach negative recall
of 0.80, then apply it to the test set and report what it costs.

In [7]:
neg_col = list(control.classes_).index(0)
oof_neg = cross_val_predict(
    control, X_train, y_train,
    cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
    method="predict_proba",
)[:, neg_col]
op_threshold = choose_threshold(y_train, oof_neg, target_recall=0.80)
op = evaluate_at_threshold(y_test, neg_test, op_threshold)
print("operating threshold (chosen on OOF train): %.3f" % op_threshold)
pd.DataFrame([
    {"threshold": "0.50 (default)", **{k: holdout[k] for k in ["neg_recall", "neg_precision", "specificity"]}},
    {"threshold": "operating point", **{k: op[k] for k in ["neg_recall", "neg_precision", "specificity"]}},
]).set_index("threshold").round(3)

operating threshold (chosen on OOF train): 0.075


,neg_recall,neg_precision,specificity
threshold,,,
0.50 (default),0.244,0.714,0.991
operating point,0.780,0.364,0.872


The precision-recall curve for the negative class, more informative than ROC under imbalance:

In [8]:
prec, rec, _ = precision_recall_curve((y_test == 0).astype(int), neg_test)
fig = px.area(x=rec, y=prec, title="Precision-recall curve (negative class), PR-AUC = %.3f" % holdout["pr_auc"],
              labels={"x": "recall", "y": "precision"})
fig.add_hline(y=(y_test == 0).mean(), line_dash="dot", annotation_text="baseline (prevalence)")
fig

The precision/recall/specificity trade-off as the threshold moves; the chosen operating point is marked:

In [9]:
sw = pd.DataFrame(sweep(y_test, neg_test))
fig = go.Figure()
for col in ["neg_recall", "neg_precision", "specificity"]:
    fig.add_trace(go.Scatter(x=sw["threshold"], y=sw[col], mode="lines", name=col))
fig.add_vline(x=op_threshold, line_dash="dot", annotation_text="operating point")
fig.update_layout(title="Threshold sweep (negative class)", xaxis_title="threshold", yaxis_title="score")
fig

## Interpretation

Explainability, separate from prediction: which words move a review toward the negative
class. The selected L2 model is dense, so we also fit an L1 model at the same `C` to show
why L1 serves explainability, a far shorter list of words carries the signal.

In [10]:
l1 = Pipeline([
    ("tfidf", build_vectorizer("tfidf", min_df=2)),
    ("clf", LogisticRegression(penalty="l1", C=10, solver="liblinear",
                               max_iter=2000, random_state=RANDOM_STATE)),
]).fit(X_train, y_train)

c2 = top_coefficients(control, n=12)
c1 = top_coefficients(l1, n=12)
print("L2: %d features, sparsity %.2f" % (c2["n_features"], c2["sparsity"]))
print("L1: %d features, sparsity %.2f (nonzero %d)"
      % (c1["n_features"], c1["sparsity"], round((1 - c1["sparsity"]) * c1["n_features"])))
pd.DataFrame({
    "L2 negative-driving": list(c2["negative_drivers"]["word"]),
    "L1 negative-driving": list(c1["negative_drivers"]["word"]),
    "L2 positive-driving": list(c2["positive_drivers"]["word"]),
})

L2: 1839 features, sparsity 0.00
L1: 1839 features, sparsity 0.85 (nonzero 268)


,L2 negative-driving,L1 negative-driving,L2 positive-driving
0,not,back,love
1,didn,awful,great
2,back,poor,easy
3,poor,stopped,like
4,no,useless,my
5,return,not,well
6,dumb,garbage,works
7,cant,didn,good
8,sucks,return,amazing
9,meh,within,an


## Fairness as separation

Separation asks whether error rates are equal across groups. Here the groups are device
`variation`, evaluated at the operating threshold: negative-class recall and false-positive
rate per group, with counts, because the rare negative class makes small groups noisy.

In [11]:
test_pred = np.where(neg_test >= op_threshold, 0, 1)
table, spread = separation_by_group(y_test.values, test_pred, test_df["variation"].values)
print("spread over groups with >= 5 negatives:", {k: round(v, 3) if isinstance(v, float) else v for k, v in spread.items()})
table.head(10)

spread over groups with >= 5 negatives: {'groups_scored': 3, 'neg_recall_spread': 0.2, 'fpr_spread': 0.018}


,variation,n,n_negatives,neg_recall,fpr
0,Black,44,6,1.000000,0.157895
1,Black Spot,49,6,0.833333,0.139535
2,Configuration: Fire TV Stick,73,5,0.800000,0.147059
3,Black Show,68,4,0.500000,0.140625
4,Black Dot,51,3,0.000000,0.062500
5,Black Plus,46,3,0.666667,0.232558
6,White,20,3,1.000000,0.000000
7,White Dot,15,3,0.666667,0.000000
8,White Plus,10,3,1.000000,0.428571
9,White Show,15,3,1.000000,0.083333


## Disagreements with the star label

The revealing cases: where the text model reads a review differently from its stars. Most
divergence is the model flagging complaint language in high-star reviews; a few are terse or
sarcastic low-star reviews the model reads as positive.

In [12]:
dis = disagreements(test_df.reset_index(drop=True), test_pred, neg_test)
print(dis["direction"].value_counts().to_dict())
show = dis.copy()
show["verified_reviews"] = show["verified_reviews"].str.slice(0, 80)
show.head(10)

{'model_flags_negative_high_star': 56, 'model_reads_positive_low_star': 9}


,direction,rating,feedback,model_pred,neg_score,variation,verified_reviews
0,model_flags_negative_high_star,3,1,0,0.750685,White Spot,I bought this to replace a Dot in my bedroom. Already have a Show and a couple o
1,model_flags_negative_high_star,5,1,0,0.558353,Black Plus,It appears to be working fine my first Echo stopped replying to my requests and
2,model_flags_negative_high_star,3,1,0,0.544965,Black Plus,Although I haven't taken advantage of many of the Echo's features there is one t
3,model_flags_negative_high_star,5,1,0,0.527995,Black,No problems
4,model_flags_negative_high_star,5,1,0,0.474314,Black Show,The only draw back is it stops every once in awhile in the middle of a song .
5,model_flags_negative_high_star,4,1,0,0.456985,Black Show,"My difficulty is I dont have a cell phone,and now you cant download the alexa ap"
6,model_flags_negative_high_star,3,1,0,0.340082,Black Plus,I can't get multiple profiles working. It just spins when adding a 2nd user at p
7,model_flags_negative_high_star,4,1,0,0.336199,Black Plus,It get on sale after 2 days so ... CHECK EVENTS BEFORE U BUY THESE AMAZON PRODUC
8,model_flags_negative_high_star,5,1,0,0.331500,Black Spot,still learning how to ise it but is DOES NOT disappoint. I’m giving to everyone
9,model_flags_negative_high_star,5,1,0,0.296094,Black,I like it because you could ask it different questions and it does different thi


## What this shows

The label is a thresholded star rating, and on this small, imbalanced dataset a
properly-selected regularized linear model is the right call: it leads the PyTorch and
TensorFlow variants on PR-AUC, so the added complexity does not earn its place. Selecting on
PR-AUC and then choosing a threshold against a stated recall target keeps the metric story
honest, and the L1 coefficients and the disagreement rows show what the model keys on and
where it is predictably wrong.